In [ ]:
get_ipython().system('pip install --upgrade pip setuptools wheel')

In [ ]:
get_ipython().system('pip install scikit-learn==1.4.2 lightgbm imbalanced-learn --only-binary=:all:')

In [ ]:
import os
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline as SkPipeline
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
from transformer import CreditScorePreprocessor, smt_strategy

SEED = 42

df = pd.read_csv("train/train.csv")
X_train = df.drop("Credit_Score", axis=1)
y_train = df["Credit_Score"]

train_pipe = ImbPipeline([
    ("preprocessing", CreditScorePreprocessor()),
    ("smote_tomek", SMOTETomek(smote=SMOTE(sampling_strategy=smt_strategy, random_state=SEED),
                               random_state=SEED)),
    ("classifier", LGBMClassifier(n_estimators=600, num_leaves=127, learning_rate=0.05,
                                  random_state=SEED, n_jobs=-1, verbose=-1)),
])
train_pipe.fit(X_train, y_train)

model = SkPipeline([
    ("preprocessing", train_pipe.named_steps["preprocessing"]),
    ("classifier", train_pipe.named_steps["classifier"]),
])

os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/model_credit.joblib")
print("✅ Training complete. Model saved to model/model_credit.joblib")